In [1]:
# Multi-Model Validation on In-The-Wild Dataset
# This notebook runs augmentation sweeps (attacks) on the In-The-Wild dataset using multiple pre-trained models (`AASIST`, `AASIST3`, `RawNet2`).

In [14]:
# Cell 1: Setup and Imports
import os, sys, csv, json, random, torch

# 1. Fix missing dependencies
try:
    import torchaudio
    import torchaudio.functional as F
    import datasets
except ImportError:
    print("Missing libraries detected. Installing...")
    !pip install torchaudio datasets pytorch-lightning wandb
    import torchaudio
    import torchaudio.functional as F

from datasets import load_dataset, Audio
from torch.utils.data import Dataset, DataLoader
from pytorch_lightning import Trainer, LightningModule
from pytorch_lightning.loggers import WandbLogger
import wandb

# 2. Patch torch.load (from validation.ipynb)
_orig_torch_load = torch.load
def _torch_load_no_weights_only(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_torch_load(*args, **kwargs)
torch.load = _torch_load_no_weights_only

# 3. Setup local project paths
sys.path.insert(0, os.getcwd())
try:
    from callbacks_rational import (
        BinaryACC_Callback, BinaryAUC_Callback, EER_Callback,
        TPR_Callback, TNR_Callback, FPR_Callback, FNR_Callback,
    )
    from loader import _crop_policy
    print("✅ Local modules (callbacks, loader) imported.")
except ImportError:
    print("❌ Critical: Could not find 'callbacks_rational.py' or 'loader.py' in the current folder.")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Environment ready on {device}")

Missing libraries detected. Installing...
Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 27.5 MB/s eta 0:00:00 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 KB 71.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 KB 47.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 43.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 35.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 KB 23.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.9/134.9 KB 31.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 642.6/642.6 KB 90.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 KB 39.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.2/193.2 KB 48.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

OSError: libcudart.so.13: cannot open shared object file: No such file or directory

In [ ]:
# Cell 2: In-The-Wild Dataset
def load_hf_token(path="secret.txt"):
    return open(path).read().strip() if os.path.exists(path) else None

HF_TOKEN = load_hf_token()

print("Loading In-The-Wild dataset (this may take a moment)...")
itw_raw = load_dataset("mueller91/In-The-Wild", token=HF_TOKEN, split='test')
itw_raw = itw_raw.cast_column("audio", Audio(sampling_rate=16000))

class ITWDataset(Dataset):
    def __init__(self, hf_split):
        self.ds = hf_split

    def __len__(self):
        return len(self.ds)

    def __getitem__(self, idx):
        item = self.ds[idx]
        wav = torch.tensor(item["audio"]["array"], dtype=torch.float32).unsqueeze(0)
        # Apply the same 4-second crop policy used for GAT
        wav = _crop_policy(wav, "eval") 
        
        # Map labels: ITW 'bonafide' is 0, 'spoof' is 1
        label_raw = item["label"]
        label = 0 if str(label_raw).lower() in ["0", "bonafide", "real"] else 1
            
        return {"audio": wav, "label": torch.tensor(label).long()}

itw_eval_set = ITWDataset(itw_raw)
print(f"✅ ITW Dataset ready ({len(itw_eval_set)} samples).")

In [11]:
# Cell 3: Pretrained Model Wrapper (The AASIST/RawNet logic)

class PretrainedWrapper(LightningModule):
    def __init__(self, model_instance, name):
        super().__init__()
        self.model = model_instance
        self.model_name = name

    def forward(self, x):
        # AASIST/RawNet expect [Batch, Length], but loader gives [Batch, 1, Length]
        if x.dim() == 3: x = x.squeeze(1)
        
        out = self.model(x)
        # Handle models that return (logits, features)
        return out[0] if isinstance(out, tuple) else out

    def validation_step(self, batch, batch_idx):
        logits = self(batch["audio"])
        
        # If model gives 2-class output, turn it into a single spoof score 
        # (Logit for 'spoof' minus logit for 'bonafide')
        if logits.dim() == 2 and logits.size(1) == 2:
            logits = logits[:, 1] - logits[:, 0]
            
        return {"logit": logits, "label": batch["label"]}

# --- LOAD MODELS ---
# IMPORTANT: You must import your actual AASIST/RawNet2 classes here
# from models.aasist import AASIST
# from models.rawnet2 import RawNet2

def init_model(model_class, weight_path, name):
    # m = model_class() 
    # m.load_state_dict(torch.load(weight_path, map_location=device))
    # return PretrainedWrapper(m, name)
    print(f"Ready to load {name} from {weight_path}")
    return None # Replace with actual initialized wrapper

# Initialize your three models
models = {
    "AASIST":  init_model(None, "aasist.pth", "AASIST"),
    "AASIST3": init_model(None, "aasist3.pth", "AASIST3"),
    "RawNet2": init_model(None, "rawnet2.pth", "RawNet2")
}

NameError: name 'LightningModule' is not defined

In [12]:
# Cell 4: Augmentation (Attack) Logic

def get_aug(name, p):
    if name == "none": return lambda x: x
    if name == "pitch": return lambda x: F.pitch_shift(x, 16000, p["n"])
    if name == "noise": return lambda x: x + torch.randn_like(x) * (10**(-p["snr"]/20))
    return lambda x: x

ATTACK_CONFIG = {
    "none": [{}],
    "pitch": [{"n": 1}, {"n": 3}],
    "noise": [{"snr": 40}, {"snr": 20}]
}

class AttackDataset(Dataset):
    def __init__(self, base, aug_fn):
        self.base = base
        self.aug_fn = aug_fn
    def __len__(self): return len(self.base)
    def __getitem__(self, i):
        item = self.base[i]
        item["audio"] = self.aug_fn(item["audio"].unsqueeze(0)).squeeze(0)
        return item

NameError: name 'Dataset' is not defined

In [13]:
# Cell 5: The Master Run Loop

RESULTS_FILE = "itw_comparative_results.csv"

for m_name, model in models.items():
    if model is None: continue
    model.to(device).eval()

    for aug_name, settings in ATTACK_CONFIG.items():
        for i, params in enumerate(settings):
            print(f">>> Testing {m_name} | Attack: {aug_name} {params}")
            
            ds = AttackDataset(itw_dataset, get_aug(aug_name, params))
            loader = DataLoader(ds, batch_size=24, num_workers=4)
            
            logger = WandbLogger(project="ITW-Attacks", name=f"{m_name}_{aug_name}_{i}")
            
            trainer = Trainer(
                accelerator="gpu" if torch.cuda.is_available() else "cpu",
                devices=1,
                logger=logger,
                callbacks=[
                    BinaryACC_Callback(batch_key="label", output_key="logit"),
                    EER_Callback(batch_key="label", output_key="logit"),
                    BinaryAUC_Callback(batch_key="label", output_key="logit")
                ]
            )
            
            res = trainer.validate(model, loader)
            
            # Save stats
            with open(RESULTS_FILE, "a") as f:
                f.write(f"{m_name},{aug_name},{json.dumps(params)},{res[0].get('val_eer')}\n")
            
            wandb.finish()

NameError: name 'models' is not defined